In [3]:
import pandas as pd
import os
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from tqdm import tqdm
import datetime

In [5]:
nace_description_path = "projects/nace_classification/nace_report_topic_analysis/data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv"
nace_descriptions = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")

In [6]:
system_prompt_format = """You are an AI assistant that generates descriptions of companies' business models as presented in annual reports, with respect to a specific industry sector definition.
You generate realistic business-related paragraphs suitable for training a text classification model.
Do NOT mention industry codes, divisions, or classifications explicitly.
"""

few_shot_prompt_format = """Here is a definition of a industry sector:

Definition: {includes} {includes_also}

{excludes}

Here are some possible subsections:
{subsections}

Here are some examples of descriptions of these classes: 
```
{gold_standard}
```

Instruction: Please generate {num_samples} paragraphs that are from this industry class.
- Write one realistic paragraph (80 to 120 words) describing business activities in the information and communication sector.
- The paragraph should focus on concrete activities, products, services, technologies, or value creation.
- Avoid generic definitions or encyclopedic language.
"""

zero_shot_prompt_format = """Here is a definition of a industry sector:

Definition: {includes} {includes_also}

{excludes}

Here are some possible subsections:
{subsections}

Instruction: Please generate {num_samples} paragraphs that are from this industry class.
- Write one realistic paragraph (80 to 120 words) describing business activities in the information and communication sector.
- The paragraph should focus on concrete activities, products, services, technologies, or value creation.
- Avoid generic definitions or encyclopedic language.
"""

In [18]:
def generate_synthetic_data(
        num_samples: int, 
        gold_standard: list, 
        includes: str,
        includes_also: str, 
        excludes: str,
        subsections: list,
        temperature: float = 0.4, 
        model: str = "gpt-4o-mini"
): 

    # Initialize LLM
    if model == "gpt-4o-mini":
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature
        )
    elif model == "openai/gpt-oss-120b":
        llm = ChatOllama(
            model=model,
            temperature=temperature, 
            base_url="http://10.80.20.127:11434/"
        )

    # Prompt
    if gold_standard == []: 
        #print("Zero Shot!")
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt_format),
            ("human", zero_shot_prompt_format)
        ])
        gold_standard_str = ""
        
    else: 
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt_format),
            ("human", few_shot_prompt_format)
        ])
        gold_standard_str = ""
        for i, text in enumerate(gold_standard): 
            gold_standard_str += f"Example {i+1}:\n{text}\n\n"
        gold_standard_str = gold_standard_str[:-2]

    # Subsections string
    subsections_str = "\n - " + "\n - ".join(subsections)

    # Adapt excludes
    if excludes != "": 
        excludes = "Excludes: " + excludes

    # Chain
    chain = prompt | llm

    # inputs
    input = {
        "num_samples": num_samples,
        "gold_standard": gold_standard_str,
        "includes": includes,
        "includes_also": includes_also,
        "excludes": excludes,
        "subsections": subsections_str,
        }

    formatted_prompt = prompt.invoke(input)

    print("Formatted Prompt:", formatted_prompt)

    # Run
    response = chain.invoke(input)

    print(response.content)

    return formatted_prompt, response.content

In [8]:
def split_synthetic_data(content: str, num_samples: int): 
    content_list = content.split("\n")
    content_list = [c for c in content_list if c != ""]
    # if len(content_list) != num_samples: 
    #     print("Warning: length of creates examples != num_samples!")
    return content_list

In [9]:
def get_sublevels(nace_class, level): 
    nace_class_temp = nace_class
    nace_id = nace_descriptions[nace_descriptions["CODE"] == nace_class_temp]["ID"].iloc[0]
    nace_class_lvl_2 = []
    nace_class_lvl_3 = []
    nace_class_lvl_4 = []

    for _, row in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id].iterrows(): 
        nace_id_temp = row["ID"]
        nace_class_lvl_2.append(f'{row["NAME"]}')
        for _, row_2 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp].iterrows(): 
            nace_id_temp_temp = row_2["ID"]
            nace_class_lvl_3.append(f'{row["NAME"]}: {row_2["NAME"]}')
            for _, row_3 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp_temp].iterrows(): 
                nace_class_lvl_4.append(f'{row["NAME"]}: {row_2["NAME"]}: {row_3["NAME"]}')
        
    if level == 2: 
        return nace_class_lvl_2
    if level == 3: 
        return nace_class_lvl_3
    if level == 4: 
        return nace_class_lvl_4

In [10]:
generate_nace_class = "A"

includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
assert includes is not None and includes != ""
includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
includes_also = "" if pd.isna(includes_also) else includes_also
excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
excludes = "" if pd.isna(excludes) else excludes

num_samples = 2
gold_standard = ["A fischeeeee", "A Weizeeeen"]
gold_standard = []

### Generate Zero-Shot Data

### Generate Few-Shot Data

In [11]:
# select gold standard data

# ds_2_desc  = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

# ds_2_desc  = ds_2_desc[pd.notna(ds_2_desc["Description"])]

# gold_standard = []
# # 1. take one of each lvl 3 class:
# for lvl_3 in ds_2_desc[pd.notna(ds_2_desc["Description"])].groupby("NACE_lvl_3").size().index: 
#     gold_standard.append(ds_2_desc[ds_2_desc["NACE_lvl_3"] == lvl_3].iloc[0])

# df_gold_standard = pd.concat(gold_standard, axis=1).T

#df_gold_standard.to_csv("/Users/hendrikweichel/Downloads/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv")

In [12]:
df_gold_standard = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

##### Hyperparams

In [13]:
level = 2
head_nace_code = "B" if level > 1 else None
generated_classes = nace_descriptions[nace_descriptions["PARENT_ID"] == head_nace_code]["CODE"]
generated_classes = ["5","6","7"]

In [14]:
# generate date 

date = datetime.datetime.now().strftime("%Y%m%d")
store_path = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_" + date + f"__level_{level}__subclasses_{head_nace_code}/"
os.makedirs(store_path, exist_ok=True)

In [15]:
generated_data = {}

In [16]:
generated_classes = ["A", "C", "J", "F", "K"]

In [19]:
num_samples = 1000
num_samples = 1
iterations_ = 50

for generate_nace_class in generated_classes:

    includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
    assert includes is not None and includes != ""
    includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
    includes_also = "" if pd.isna(includes_also) else includes_also
    excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
    excludes = "" if pd.isna(excludes) else excludes

    gold_standard = df_gold_standard[df_gold_standard["NACE_letter"] == generate_nace_class]["Description_clean"].to_list()[:3]
    
    subsections = get_sublevels(generate_nace_class, level=2)

    examples = ""

    for i in tqdm(range(iterations_), desc=generate_nace_class):
        res = generate_synthetic_data(num_samples=num_samples, gold_standard=gold_standard, includes=includes, includes_also=includes_also, excludes=excludes, subsections=subsections, model="openai/gpt-oss-120b")
        examples += res[1]
        data = split_synthetic_data(examples, num_samples * iterations_)
        pd.DataFrame(data, columns=[generate_nace_class]).to_csv(os.path.join(store_path, f"class_{generate_nace_class}.csv"), index=False)
    
    results = {
        "data": data,
        "prompt": res[0],
        "system_prompt": res[0].messages[0].content,
        "user_prompt": res[0].messages[1].content,
        "output": examples
    }

    generated_data[generate_nace_class] = results

A:   0%|                                                                                                                                                                                       | 0/50 [00:00<?, ?it/s]


Formatted Prompt: messages=[SystemMessage(content="You are an AI assistant that generates descriptions of companies' business models as presented in annual reports, with respect to a specific industry sector definition.\nYou generate realistic business-related paragraphs suitable for training a text classification model.\nDo NOT mention industry codes, divisions, or classifications explicitly.\n", additional_kwargs={}, response_metadata={}), HumanMessage(content="Here is a definition of a industry sector:\n\nDefinition: This section includes the exploitation of vegetal and animal natural resources, comprising the activities of growing of crops, raising and breeding of animals, harvesting of timber and other plants, animals or animal products from a farm or their natural habitats. \n\n\n\nHere are some possible subsections:\n\n - Crop and animal production, hunting and related service activities\n - Forestry and logging\n - Fishing and aquaculture\n\nHere are some examples of descriptio

ConnectError: [Errno 111] Connection refused

In [ ]:
# print prompts
for k,v in generated_data.items(): 
    print(k,"_______"*20)
    print(v["user_prompt"])


1 ____________________________________________________________________________________________________________________________________________
Here is a definition of a industry sector:

Definition: This division includes two basic activities, namely the production of crop products and production of animal products, covering also the forms of organic agriculture, the growing of genetically modified crops and the raising of genetically modified animals. This division includes growing of crops in open fields as well in greenhouses.\n \nGroup 01.5 (Mixed farming) breaks with the usual principles for identifying main activity. It accepts that many agricultural holdings have reasonably balanced crop and animal production, and that it would be arbitrary to classify them in one category or the other. This division also includes service activities incidental to agriculture, as well as hunting, trapping and related activities.

Excludes: Agricultural activities exclude any subsequent processing

#### aggregate data and split

In [ ]:
# config

config = {
    "prompts": {k: v["user_prompt"] for k, v in generated_data.items()}, 
    "samples": num_samples * iterations_,
    "generated_iterations": iterations_,
    "level": level,
    "head_nace_code": head_nace_code,
    "system_prompt": system_prompt_format
}

# store
import json
with open(os.path.join(store_path, "config.json"), "w") as f: 
    json.dump(config, f, indent=4)

In [ ]:
df_full = []
for k, v in generated_data.items(): 
    df_temp = pd.DataFrame(v["data"], columns=["text"])
    df_temp["label"] = k
    df_full.append(df_temp)
df_full = pd.concat(df_full, axis=0)

In [ ]:
# make new index from 0 to len(df_full)-1
df_full = df_full.reset_index(drop=True)
df_full

,text,label
0,1. The company specializes in providing cloud-...,A
1,2. As a leading provider of digital marketing ...,A
2,3. The company operates a state-of-the-art dat...,A
3,"4. Focusing on mobile technology, the company ...",A
4,5. The company is a pioneer in the field of cy...,A
...,...,...
2250,6. We operate a robust online marketplace that...,K
2251,7. Our organization provides innovative pensio...,K
2252,8. We are a fintech company that specializes i...,K
2253,9. Our company focuses on delivering comprehen...,K


In [ ]:
import re
clean_text = lambda x: re.sub(r'^\d+\.\s*', " ", x).strip()

In [ ]:
df_full["text"] = df_full["text"].apply(clean_text)

In [ ]:
df_full.to_csv(os.path.join(store_path, "synthetic_data_full.csv"), index=False)

In [ ]:
# make train test split 6:2:2

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_full, test_size=0.4, random_state=42, stratify=df_full["label"])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

len(train_df), len(test_df), len(val_df)

(1353, 451, 451)

In [ ]:
train_df.to_csv(os.path.join(store_path, "train_data.csv"), index=False)
test_df.to_csv(os.path.join(store_path, "test_data.csv"), index=False)
val_df.to_csv(os.path.join(store_path, "val_data.csv"), index=False)

In [4]:
llm = ChatOllama(
            model="llama3.1:8b-instruct-fp16",
            temperature=0.1, 
            base_url="http://10.80.20.101:11434/"
        )

In [5]:
llm.invoke("HI")

ResponseError: llama runner process has terminated: CUDA error: out of memory
  current device: 0, in function ggml_backend_cuda_device_get_memory at //ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:3238
  cudaMemGetInfo(free, total)
//ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:84: CUDA error

In [7]:
import tiktoken

tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")

In [10]:
len(tokenizer.encode("""1. Our company specializes in sustainable logging practices, focusing on the selective harvesting of timber from both natural and planted forests. We employ advanced techniques to minimize environmental impact while maximizing yield. Our operations include the use of eco-friendly machinery that reduces soil disturbance and promotes forest regeneration. Additionally, we provide firewood and charcoal products sourced from responsibly managed forests, ensuring that our offerings meet high environmental standards. By collaborating with local communities, we also support the gathering of non-wood forest products, enhancing biodiversity and fostering economic development in rural areas.

2. In our forestry operations, we prioritize silviculture techniques that enhance forest health and productivity. We implement practices such as thinning and controlled burns to promote the growth of high-quality timber. Our team conducts regular assessments to monitor forest conditions and adapt our management strategies accordingly. We also engage in the collection of non-wood products like wild mushrooms and medicinal herbs, which are harvested sustainably to ensure long-term availability. By integrating these activities, we create a diversified revenue stream while contributing to the conservation of forest ecosystems.

3. Our logging company is committed to responsible forest management, focusing on the extraction of roundwood in a way that preserves the ecological balance of the forest. We utilize state-of-the-art equipment designed to minimize waste and ensure the efficient harvesting of timber. Our operations are complemented by a robust training program for our workforce, emphasizing safety and environmental stewardship. Furthermore, we offer a range of products, including pit-props and pulpwood, which are supplied to various industries, ensuring that our timber is used effectively and sustainably.

4. We are dedicated to the gathering of wild growing non-wood forest products, which play a vital role in supporting local economies and promoting biodiversity. Our team works closely with foragers to identify and harvest edible plants, berries, and nuts in a sustainable manner. By implementing strict guidelines on harvesting practices, we ensure that these resources are available for future generations. Additionally, we provide training and resources to local communities, empowering them to participate in this industry while preserving their traditional knowledge and practices.

5. Our company offers comprehensive support services to forestry operations, including consulting on sustainable practices and forest management planning. We provide expertise in areas such as reforestation, pest management, and soil conservation, helping clients optimize their forestry activities. Our services extend to training programs for forest workers, focusing on safety protocols and sustainable harvesting techniques. By partnering with clients, we aim to enhance the productivity and sustainability of their forestry operations, contributing to the overall health of forest ecosystems.

6. We focus on the extraction of high-quality roundwood, utilizing advanced logging techniques that prioritize sustainability and efficiency. Our operations are designed to minimize waste and enhance the recovery of valuable timber products. We also engage in the production of firewood, which is sourced from our managed forests and processed to meet consumer demand. Our commitment to responsible forestry practices ensures that we not only meet market needs but also contribute positively to the environment and local communities.

7. Our forestry management firm specializes in the cultivation and maintenance of planted forests, ensuring a steady supply of timber for various applications. We implement innovative silvicultural practices that enhance growth rates and timber quality, while also focusing on biodiversity conservation. In addition to timber production, we engage in the collection of non-wood forest products, such as wild herbs and berries, which are marketed to local businesses. This dual approach allows us to maximize the economic value of our forests while promoting ecological health.

8. We are involved in the logging sector, where our operations emphasize the careful extraction of timber from both natural and managed forests. Our commitment to sustainability is reflected in our use of low-impact logging techniques, which help preserve the integrity of the forest ecosystem. We also produce charcoal and firewood, catering to the growing demand for renewable energy sources. By maintaining a focus on environmental responsibility, we strive to balance economic viability with ecological preservation.

9. Our company is dedicated to the sustainable gathering of wild growing non-wood products, which are integral to the livelihoods of many local communities. We prioritize ethical harvesting practices that ensure the long-term availability of these resources. Our team collaborates with local foragers to promote best practices and provide training on sustainable collection techniques. By creating a market for these products, we not only support local economies but also contribute to the conservation of forest biodiversity.

10. We provide essential support services to forestry operations, helping clients navigate the complexities of sustainable forest management. Our team offers expertise in areas such as land assessment, resource inventory, and compliance with environmental regulations. We also facilitate training programs focused on best practices for logging and non-wood product harvesting. By equipping forestry businesses with the knowledge and tools they need, we aim to enhance their operational efficiency and promote sustainable practices across the sector.

"""))

950